# Importing all Necessary Libraries

This section imports all the libraries required for data manipulation, visualization, and exploratory data analysis.

In [ ]:
# Data Manipulation

import pandas as pd
import numpy as np

# Visualization

import matplotlib.pyplot as plt
import seaborn as sns

# Ignore warnings

import warnings
warnings.filterwarnings("ignore")

# Plot settings

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12,6)
plt.rcParams["font.size"] = 12

# Display all columns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Loading the Dataset

In [ ]:
# Load datasets
demand_df = pd.read_csv("../data/raw/demand.csv")
promo_df = pd.read_csv("../data/raw/promotions.csv")

# Dataset Overview

This section provides an initial understanding of the datasets by examining their dimensions, sample records, data types, and summary statistics.

In [ ]:
print("Demand Dataset Shape:", demand_df.shape)
print("Promotion Dataset Shape:", promo_df.shape)

In [ ]:
demand_df.head()

In [ ]:
promo_df.head()

In [ ]:
demand_df.sample(5)

In [ ]:
demand_df.info()

In [ ]:
promo_df.info()

In [ ]:
demand_df.describe()

In [ ]:
promo_df.describe()

In [ ]:
demand_df.describe(include="object")

# Missing Value Analysis

The objective of this section is to identify missing values within the datasets and understand their distribution before performing any preprocessing.

In [ ]:
missing = demand_df.isnull().sum()

missing

In [ ]:
missing = promo_df.isnull().sum()

missing

In [ ]:
missing_percent = (
    demand_df.isnull().mean()*100
).sort_values(ascending=False)

missing_percent

In [ ]:
plt.figure(figsize=(8,4))

sns.barplot(
    x=missing_percent.index,
    y=missing_percent.values
)

plt.title("Missing Values (%)")
plt.ylabel("Percentage")

plt.show()

In [ ]:
plt.savefig("../outputs/figures/missing_values.png")

# Duplicate Record Analysis

Duplicate observations can introduce bias during model training. This section verifies whether duplicate records exist in either dataset.

In [ ]:
duplicates = demand_df.duplicated().sum()

print("Duplicate Rows:", duplicates)

In [ ]:
duplicates = promo_df.duplicated().sum()
print("Duplicate Rows:", duplicates)

# Date Range Validation

The date columns are converted to datetime format to verify the temporal coverage of the datasets and prepare them for time-series analysis.

In [ ]:
demand_df["date"] = pd.to_datetime(demand_df["date"])
promo_df["promotion_date"] = pd.to_datetime(
    promo_df["promotion_date"]
)

In [ ]:
demand_start = demand_df["date"].min()
demand_end = demand_df["date"].max()

print("Demand Start:", demand_start)
print("Demand End:", demand_end)

In [ ]:
outside_promotions = promo_df[
    (promo_df["promotion_date"] < demand_start) |
    (promo_df["promotion_date"] > demand_end)
]

outside_promotions

# Promotion Data Analysis

This section explores promotional campaigns across supermarkets and products to understand their distribution and potential influence on product demand.

In [ ]:
promo_df["supermarket"].value_counts()

In [ ]:
promo_df["sku"].value_counts()

In [ ]:
promotion_summary = pd.crosstab(
    promo_df["supermarket"],
    promo_df["sku"]
)

promotion_summary

# Demand Distribution Analysis

Understanding the distribution of demand helps identify skewness, variability, and potential outliers that may influence forecasting performance.

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    data=demand_df,
    x="demand",
    bins=30,
    kde=True
)

plt.title("Distribution of Demand")
plt.xlabel("Demand")
plt.ylabel("Frequency")

plt.savefig("../outputs/figures/demand_distribution.png")
plt.show()

In [ ]:
plt.figure(figsize=(10,2))

sns.boxplot(
    x=demand_df["demand"]
)

plt.title("Boxplot of Demand")

plt.savefig("../outputs/figures/demand_boxplot.png")
plt.show()

In [ ]:
print("Mean:", demand_df["demand"].mean())
print("Median:", demand_df["demand"].median())
print("Standard Deviation:", demand_df["demand"].std())
print("Skewness:", demand_df["demand"].skew())
print("Kurtosis:", demand_df["demand"].kurt())

# Missing Demand Analysis by Supermarket and Product

Missing demand values are analyzed across supermarkets and products to determine whether the missing data follows any systematic pattern.

In [ ]:
plt.figure(figsize=(15,3))

sns.heatmap(
    demand_df.isnull(),
    cbar=False,
    yticklabels=False
)

plt.title("Missing Values Heatmap")

plt.savefig("../outputs/figures/missing_values_heatmap.png")
plt.show()

In [ ]:
missing_supermarket = (
    demand_df.groupby("supermarket")["demand"]
    .apply(lambda x: x.isnull().sum())
    .sort_values(ascending=False)
)

missing_supermarket

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    x=missing_supermarket.index,
    y=missing_supermarket.values
)

plt.title("Missing Demand by Supermarket")
plt.xlabel("Supermarket")
plt.ylabel("Missing Values")

plt.savefig("../outputs/figures/missing_by_supermarket.png")
plt.show()

In [ ]:
missing_sku = (
    demand_df.groupby("sku")["demand"]
    .apply(lambda x: x.isnull().sum())
    .sort_values(ascending=False)
)

missing_sku

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    x=missing_sku.index,
    y=missing_sku.values
)

plt.title("Missing Demand by SKU")
plt.xlabel("SKU")
plt.ylabel("Missing Values")

plt.savefig("../outputs/figures/missing_by_sku.png")
plt.show()

In [ ]:
# Display observations with unusually high demand
high_demand = demand_df[demand_df["demand"] > 200].sort_values("demand", ascending=False)

high_demand

In [ ]:
# Aggregate demand across all supermarkets and SKUs
daily_demand = (
    demand_df
    .groupby("date")["demand"]
    .sum()
    .reset_index()
)

daily_demand.head()

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(
    daily_demand["date"],
    daily_demand["demand"],
    linewidth=1
)

plt.title("Total Daily Demand Over Time")
plt.xlabel("Date")
plt.ylabel("Total Demand")

plt.tight_layout()
plt.savefig("../outputs/figures/daily_demand_trend.png")
plt.show()

# Time-Series Trend Analysis

Daily demand is aggregated across all supermarkets and products to visualize long-term demand trends and seasonality.

In [ ]:
daily_demand["rolling_30"] = (
    daily_demand["demand"]
    .rolling(window=30)
    .mean()
)

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(
    daily_demand["date"],
    daily_demand["demand"],
    alpha=0.4,
    label="Daily Demand"
)

plt.plot(
    daily_demand["date"],
    daily_demand["rolling_30"],
    linewidth=2,
    label="30-Day Rolling Average"
)

plt.title("Daily Demand with 30-Day Rolling Average")
plt.xlabel("Date")
plt.ylabel("Demand")

plt.legend()

plt.tight_layout()
plt.savefig("../outputs/figures/daily_trend_rolling_average.png")
plt.show()

# Monthly Demand Analysis

Monthly demand patterns are examined to identify recurring seasonal behaviour throughout the year.

In [ ]:
monthly_demand = (
    demand_df
    .set_index("date")
    .resample("ME")["demand"]
    .sum()
    .reset_index()
)

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(
    monthly_demand["date"],
    monthly_demand["demand"],
    marker="o"
)

plt.title("Average Monthly Demand")
plt.xlabel("Month")
plt.ylabel("Average Daily Demand")

plt.tight_layout()
plt.savefig("../outputs/figures/monthly_demand_trend.png")
plt.show()

In [ ]:
sku_daily = (
    demand_df
    .groupby(["date", "sku"])["demand"]
    .sum()
    .reset_index()
)

sku_daily.head()

In [ ]:
plt.figure(figsize=(15,6))

for sku in sku_daily["sku"].unique():
    temp = sku_daily[sku_daily["sku"] == sku]

    plt.plot(
        temp["date"],
        temp["demand"],
        label=sku,
        linewidth=2
    )

plt.title("Daily Demand Trend by SKU")
plt.xlabel("Date")
plt.ylabel("Total Daily Demand")
plt.legend()

plt.tight_layout()
plt.savefig("../outputs/figures/demand_by_sku.png")
plt.show()

# Supermarket-Level Demand Analysis

This section compares demand patterns across supermarkets to understand differences in customer purchasing behaviour.

In [ ]:
market_daily = (
    demand_df
    .groupby(["date", "supermarket"])["demand"]
    .sum()
    .reset_index()
)

In [ ]:
plt.figure(figsize=(15,6))

for market in market_daily["supermarket"].unique():

    temp = market_daily[
        market_daily["supermarket"] == market
    ]

    plt.plot(
        temp["date"],
        temp["demand"],
        label=market,
        linewidth=2
    )

plt.title("Daily Demand Trend by Supermarket")
plt.xlabel("Date")
plt.ylabel("Total Daily Demand")
plt.legend()

plt.tight_layout()
plt.savefig("../outputs/figures/demand_by_supermarket.png")
plt.show()

In [ ]:
sku_daily["rolling_30"] = (
    sku_daily
    .groupby("sku")["demand"]
    .transform(lambda x: x.rolling(30, min_periods=1).mean())
)

In [ ]:
plt.figure(figsize=(15,6))

for sku in sku_daily["sku"].unique():

    temp = sku_daily[
        sku_daily["sku"] == sku
    ]

    plt.plot(
        temp["date"],
        temp["rolling_30"],
        linewidth=3,
        label=sku
    )

plt.title("30-Day Rolling Average Demand by SKU")
plt.xlabel("Date")
plt.ylabel("Average Demand")
plt.legend()

plt.tight_layout()
plt.savefig("../outputs/figures/rolling_demand_by_sku.png")
plt.show()

In [ ]:
market_daily["rolling_30"] = (
    market_daily
    .groupby("supermarket")["demand"]
    .transform(lambda x: x.rolling(30, min_periods=1).mean())
)

In [ ]:
plt.figure(figsize=(15,6))

for market in market_daily["supermarket"].unique():

    temp = market_daily[
        market_daily["supermarket"] == market
    ]

    plt.plot(
        temp["date"],
        temp["rolling_30"],
        linewidth=3,
        label=market
    )

plt.title("30-Day Rolling Average Demand by Supermarket")
plt.xlabel("Date")
plt.ylabel("Average Demand")
plt.legend()

plt.tight_layout()
plt.savefig("../outputs/figures/rolling_demand_by_supermarket.png")
plt.show()

# Weekly Demand Pattern

Demand is analyzed across different days of the week to identify recurring weekly seasonality.

In [ ]:
# Create day of week feature
demand_df["day_of_week"] = demand_df["date"].dt.day_name()

In [ ]:
weekday_demand = (
    demand_df
    .groupby("day_of_week")["demand"]
    .mean()
    .reindex([
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday"
    ])
)

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(
    x=weekday_demand.index,
    y=weekday_demand.values
)

plt.title("Average Demand by Day of Week")
plt.xlabel("Day")
plt.ylabel("Average Demand")

plt.tight_layout()

plt.savefig("../outputs/figures/weekday_demand.png")

plt.show()

In [ ]:
demand_df["month"] = demand_df["date"].dt.month_name()

In [ ]:
month_order = [
    "January","February","March","April",
    "May","June","July","August",
    "September","October","November","December"
]

monthly_avg = (
    demand_df
    .groupby("month")["demand"]
    .mean()
    .reindex(month_order)
)

In [ ]:
plt.figure(figsize=(12,5))

sns.barplot(
    x=monthly_avg.index,
    y=monthly_avg.values
)

plt.xticks(rotation=45)

plt.title("Average Demand by Month")
plt.xlabel("Month")
plt.ylabel("Average Demand")

plt.tight_layout()

plt.savefig("../outputs/figures/monthly_seasonality.png")

plt.show()

In [ ]:
promo = promo_df.copy()

In [ ]:
promo.drop(columns=["Unnamed: 0"], inplace=True)

In [ ]:
promo["promotion_end"] = (
    promo["promotion_date"]
    + pd.Timedelta(days=6)
)

In [ ]:
demand_df["promotion"] = 0

In [ ]:
for _, row in promo.iterrows():

    mask = (
        (demand_df["date"] >= row["promotion_date"]) &
        (demand_df["date"] <= row["promotion_end"]) &
        (demand_df["sku"] == row["sku"]) &
        (demand_df["supermarket"] == row["supermarket"])
    )

    demand_df.loc[mask, "promotion"] = 1

In [ ]:
promotion_effect = (
    demand_df
    .groupby("promotion")["demand"]
    .mean()
)

promotion_effect

In [ ]:
plt.figure(figsize=(6,5))

sns.barplot(
    x=["No Promotion","Promotion"],
    y=promotion_effect.values
)

plt.title("Average Demand During Promotion")

plt.ylabel("Average Demand")

plt.tight_layout()

plt.savefig("../outputs/figures/promotion_effect.png")

plt.show()

# Correlation Analysis

Correlation analysis is performed on numerical variables to identify relationships between features that may influence future demand.

In [ ]:
pivot = demand_df.pivot_table(
    values="demand",
    index="supermarket",
    columns="sku",
    aggfunc="mean"
)

plt.figure(figsize=(8,5))

sns.heatmap(
    pivot,
    annot=True,
    cmap="YlGnBu",
    fmt=".1f"
)

plt.title("Average Demand by Supermarket and SKU")

plt.tight_layout()

plt.savefig("../outputs/figures/demand_heatmap.png")

plt.show()

In [ ]:
corr = demand_df.select_dtypes(include="number").corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr,annot=True,cmap="coolwarm")

# Demand Variability

Boxplots are used to visualize the spread of demand values and identify potential outliers.

In [ ]:
plt.boxplot(demand_df["demand"].dropna())

In [ ]:
sns.boxplot(
    x="sku",
    y="demand",
    data=demand_df
)

# Exploratory Data Analysis Summary

The objective of this exploratory data analysis was to understand the historical demand data, identify key demand drivers, assess data quality, and discover patterns that would guide feature engineering and model selection for the demand forecasting task.

## Data Quality

* The demand dataset contains 9,855 daily observations across 3 supermarket chains and 3 product SKUs.
* Approximately 11.3% of demand values are missing, while no duplicate records were found.
* Missing values are evenly distributed across supermarkets and SKUs rather than concentrated in specific periods, suggesting that they can be handled using appropriate imputation techniques during preprocessing.



## Demand Distribution

* Most daily demand values fall between 50 and 100 units.
* The demand distribution is slightly right-skewed due to a few high-demand observations.
* Several extreme demand values were identified; however, these occurred repeatedly for specific supermarket-SKU combinations across multiple years, indicating that they are genuine business events rather than erroneous records.



## Demand Trend

* Overall demand exhibits a gradual upward trend throughout the three-year period.
* Daily demand shows significant short-term fluctuations, while the 30-day rolling average reveals steady long-term business growth.
* Monthly average demand also indicates a gradual increase without major structural changes.



## Product-Level Insights

* Organic Milk consistently records the highest demand among all products.
* Whole Wheat Bread demonstrates moderate demand with stable growth.
* Free Range Eggs has the lowest average demand while maintaining a gradual upward trend.

These observations indicate that product type is an important factor influencing demand.



## Supermarket-Level Insights

* DailyNeeds consistently records the highest average demand.
* GreenBasket follows with moderate demand.
* FreshMart has the lowest average demand among the three supermarkets.

This suggests that customer purchasing behavior differs across supermarket chains, making supermarket identity an important predictive feature.



## Seasonality Analysis

Weekly seasonality is relatively weak, with only small differences in demand across weekdays. Similarly, monthly demand remains fairly stable throughout the year, indicating limited annual seasonality.

Although these temporal patterns are not strong, they may still provide useful predictive information and will therefore be included as engineered features.



## Promotion Analysis

Demand during promotional periods is noticeably higher than during non-promotional periods, with an average increase of approximately 9–10%. This confirms that promotions have a measurable impact on customer purchasing behavior and should be incorporated into the forecasting model.


## Key Demand Drivers

Based on the exploratory analysis, the primary factors influencing demand appear to be:

* Historical demand trend
* Product (SKU)
* Supermarket
* Promotional events

## Secondary factors include:

* Day of the week
* Month of the year


## Implications for Feature Engineering

The insights obtained during EDA will guide the next stage of the project. The forecasting model will incorporate both historical and calendar-based information by engineering features such as:

* Lag demand features
* Rolling statistics
* Promotion indicator
* Day of week
* Month
* Product (SKU)
* Supermarket

These features will enable the model to capture long-term trends, temporal patterns, and business-specific demand drivers, leading to more accurate demand forecasts.